# Exponential distribution: optimizer comparison

Distortion vs iteration for the optimizers you choose (**Lloyd**, **mean-field CLVQ**, **Newton–Raphson**, **Newton–Raphson (LM)**), starting from the **same** random centroids. Keys in code: `lloyd`, `mfclvq`, `nr`, `nrlm`.

Use **`n_iter >= 20`** when including **`nr`** or **`nrlm`**: both run 20 Lloyd warmup iterations, then `n_iter - 20` main steps, so the distortion series has length `n_iter`.

Run this notebook from the **repository root** (or set the kernel’s working directory there) so `univariate` imports resolve.

In [6]:
import sys
from pathlib import Path

# Ensure the repository root is on sys.path (works from repo root or from notebooks/)
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np

from bokeh.io import output_notebook, show
from bokeh.palettes import Category10
from bokeh.plotting import figure

from univariate.exponential_quantization import ExponentialVoronoiQuantization
from univariate.demos.optimizer_comparison import compare_methods, compare_nrlm_sweep

output_notebook()

np.random.seed(0)
N = 100
n_iter = 100

# Choose which methods to compare: any subset of ["lloyd", "mfclvq", "nr", "nrlm"]
methods = ["lloyd", "mfclvq", "nr", "nrlm"]

quantizer = ExponentialVoronoiQuantization()
initial = np.random.exponential(size=N)
curves = compare_methods(
    quantizer,
    initial,
    n_iter,
    methods=methods,
    nrlm_num_warmup_iterations=5,
)

p = figure(
    title="Exponential: distortion vs iteration (same initial centroids)",
    width=850,
    height=480,
    x_axis_label="Iteration",
    y_axis_label="Distortion",
    toolbar_location="above",
)

colors = Category10[10]
for i, (label, d) in enumerate(curves.items()):
    x = list(range(1, len(d) + 1))
    p.line(x, d, line_width=2, color=colors[i % len(colors)], legend_label=label)

p.legend.location = "top_right"
p.legend.click_policy = "hide"
p.grid.grid_line_alpha = 0.25

show(p)

Loading BokehJS ...

2026-04-30 15:16:18.559 | INFO     | univariate.voronoi_quantization:deterministic_lloyd_method:113 - Start Lloyd (N=100, iterations=100)
2026-04-30 15:16:18.562 | INFO     | univariate.voronoi_quantization:deterministic_lloyd_method:128 - End Lloyd (final_distortion=0.0043189766973376065)
2026-04-30 15:16:18.562 | INFO     | univariate.voronoi_quantization:mean_field_clvq_method:136 - Start mean-field CLVQ (N=100, iterations=100)
2026-04-30 15:16:18.565 | INFO     | univariate.voronoi_quantization:mean_field_clvq_method:158 - End mean-field CLVQ (final_distortion=0.011285349608309736)
2026-04-30 15:16:18.566 | INFO     | univariate.voronoi_quantization:newton_raphson_method:167 - Start Newton–Raphson (warmup_lloyd=20, iterations=100, N=100)
2026-04-30 15:16:18.566 | INFO     | univariate.voronoi_quantization:deterministic_lloyd_method:113 - Start Lloyd (N=100, iterations=20)
2026-04-30 15:16:18.566 | INFO     | univariate.voronoi_quantization:deterministic_lloyd_method:128 - End Lloyd

In [8]:
# NR+LM hyperparameter sweep
lambda_0_values = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]

sweep = compare_nrlm_sweep(
    quantizer,
    initial,
    n_iter=20,
    lambda_0_values=lambda_0_values,
    diagonal_term_types=("identity", "hessian"),
    nrlm_num_warmup_iterations=5,
)

p2 = figure(
    title="Exponential: NR+LM sweep over λ0 and diagonal term",
    width=850,
    height=480,
    x_axis_label="Iteration",
    y_axis_label="Distortion",
    toolbar_location="above",
)

colors = Category10[10]
dash_for_diag = {"identity": "solid", "hessian": "dashed"}

for lam_idx, lam0 in enumerate(lambda_0_values):
    color = colors[lam_idx % len(colors)]
    for diag in ("identity", "hessian"):
        d = sweep[(diag, float(lam0))]
        x = list(range(1, len(d) + 1))
        p2.line(
            x,
            d,
            line_width=2,
            color=color,
            line_dash=dash_for_diag[diag],
            legend_label=f"{diag}  λ0={lam0:g}",
        )

p2.legend.location = "top_right"
p2.legend.click_policy = "hide"
p2.grid.grid_line_alpha = 0.25

show(p2)

2026-04-30 15:16:31.517 | INFO     | univariate.voronoi_quantization:newton_raphson_method_with_levenberg_marquardt:198 - Start NR+LM (warmup_lloyd=5, iterations=20, N=100, lambda_0=0.0001, diagonal_term_type=identity)
2026-04-30 15:16:31.517 | INFO     | univariate.voronoi_quantization:deterministic_lloyd_method:113 - Start Lloyd (N=100, iterations=5)
2026-04-30 15:16:31.518 | INFO     | univariate.voronoi_quantization:deterministic_lloyd_method:128 - End Lloyd (final_distortion=0.06626855004575738)
2026-04-30 15:16:31.522 | INFO     | univariate.voronoi_quantization:newton_raphson_method_with_levenberg_marquardt:263 - End NR+LM (final_distortion=0.0001358349624469568)
2026-04-30 15:16:31.522 | INFO     | univariate.voronoi_quantization:newton_raphson_method_with_levenberg_marquardt:198 - Start NR+LM (warmup_lloyd=5, iterations=20, N=100, lambda_0=0.001, diagonal_term_type=identity)
2026-04-30 15:16:31.522 | INFO     | univariate.voronoi_quantization:deterministic_lloyd_method:113 - S

In [3]:
for label, d in curves.items():
    print(f"{label:20s} final distortion = {d[-1]:.8f}")

Lloyd                final distortion = 0.01018412
Mean-field CLVQ      final distortion = 0.04061158
Newton–Raphson       final distortion = 0.01009439
Newton–Raphson (LM)  final distortion = 0.01009439
